# Train the NAT decision tree

This notebook trains the runtime `NATClassifier` with a depth-8 decision tree. The classifier owns the pandas aggregation, so training and DAF execution both follow the same path:

```text
one IP in one time window -> NATClassifier.aggregate() -> decision tree
```

The source CSV is large and ordered by capture segment. We take an equal number of non-NAT rows from its beginning and NAT rows from its end without loading the complete file. The final model table is balanced again after IP/window aggregation.

In [ ]:
from pathlib import Path
import io
import sys

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").exists()
)
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from detectors.common import TimeWindowedIPFlowDataset
from detectors.nat_detector import NATClassifier

## 1. Configure the training run

`window_size` must match the DAF runtime window size used with this model. The sample size keeps this notebook practical on a workstation while still providing thousands of distinct source IPs per class.

In [ ]:
training_dir = repo_root / "src" / "detectors" / "nat_detector" / "training"
data_path = training_dir / "data" / "combined_NAT_anonym.csv"
model_path = (
    repo_root
    / "src/detectors/nat_detector/final_models/decision_tree"
    / "nat_decision_tree_depth_8.joblib"
)

src_ip_field = "SRC_IP"
timestamp_column = "TIME_FIRST"
label_column = "IS_NAT"
window_size = "15min"
rows_per_class = 100_000
random_state = 42

if not data_path.is_file():
    raise FileNotFoundError(f"Training data not found: {data_path}")

## 2. Define the runtime aggregation

These are pandas named aggregations. They use no notebook-local callables, which keeps the saved classifier portable. The same `classifier.aggregate` method is applied to every IP/window below and later called by `nat_detector.py` during execution.

In [ ]:
nat_aggregation = {
    "source_port_count": ("SRC_PORT", "nunique"),
    "destination_port_count": ("DST_PORT", "nunique"),
    "tcp_syn_size_count": ("TCP_SYN_SIZE", "nunique"),
    "tcp_window_count": ("TCP_WIN", "nunique"),
    "ttl_count": ("IP_TTL", "nunique"),
    "mean_ttl": ("IP_TTL", "mean"),
    "mean_source_port": ("SRC_PORT", "mean"),
    "mean_destination_port": ("DST_PORT", "mean"),
    "total_bytes": ("BYTES", "sum"),
    "total_reverse_bytes": ("BYTES_REV", "sum"),
    "total_packets": ("PACKETS", "sum"),
    "total_reverse_packets": ("PACKETS_REV", "sum"),
    "flow_count": (src_ip_field, "size"),
}

classifier = NATClassifier(
    model=DecisionTreeClassifier(
        max_depth=8,
        class_weight="balanced",
        random_state=random_state,
    ),
    aggregation=nat_aggregation,
    positive_label=1,
    threshold=0.5,
)

columns_to_load = sorted(
    {
        src_ip_field,
        timestamp_column,
        label_column,
        *(column for column, _ in nat_aggregation.values()),
    }
)

## 3. Read dataset

In [ ]:
non_nat_flows = pd.read_csv(
    data_path_no_nat,
    usecols=columns_to_load,
)
nat_flows = pd.read_csv(
    data_path_nat,
    usecols=columns_to_load,
)

label_values = {
    "false": 0,
    "0": 0,
    "true": 1,
    "1": 1,
}
for frame in (non_nat_flows, nat_flows):
    normalized_labels = frame[label_column].astype(str).str.strip().str.lower()
    frame[label_column] = normalized_labels.map(label_values).astype("int8")

if not non_nat_flows[label_column].eq(0).all():
    raise ValueError("The first sampled rows are not exclusively non-NAT")
if not nat_flows[label_column].eq(1).all():
    raise ValueError("The last sampled rows are not exclusively NAT")

sampled_flows = pd.concat([non_nat_flows, nat_flows], ignore_index=True)
sampled_flows[label_column].value_counts().sort_index()

## 4. Build one feature row per IP and time window

`TimeWindowedIPFlowDataset` owns timestamp parsing and bucket assignment. Applying `classifier.aggregate` here guarantees that every training row is prepared in the same way as one runtime NAT decision. Labels are kept outside the model features and checked for consistency within each IP/window.

In [ ]:
windowed_dataset = TimeWindowedIPFlowDataset(
    sampled_flows,
    timestamp_column=timestamp_column,
    window_size=window_size,
    src_ip_field=src_ip_field,
)
feature_matrix = windowed_dataset.apply(classifier.aggregate)

label_summary = pd.concat(
    {
        window_start: ip_dataset.agg(
            is_nat=(label_column, "first"),
            distinct_labels=(label_column, "nunique"),
        )
        for window_start, ip_dataset in windowed_dataset
    },
    names=["window_start", src_ip_field],
)
if not label_summary["distinct_labels"].eq(1).all():
    raise ValueError("An IP/window contains conflicting NAT labels")
if feature_matrix.isna().any().any():
    raise ValueError("The selected aggregation produced missing feature values")

model_table = feature_matrix.join(label_summary["is_nat"])
samples_per_class = int(model_table["is_nat"].value_counts().min())
balanced_table = (
    model_table.groupby("is_nat", group_keys=False)
    .sample(n=samples_per_class, random_state=random_state)
    .sort_index()
)
balanced_table["is_nat"].value_counts().sort_index()

## 5. Split by source IP

A row-level random split could put different windows for the same anonymized IP in both partitions. One fold of `StratifiedGroupKFold` gives an approximately 80/20 stratified split while keeping every source IP entirely in either training or test data.

In [ ]:
X = balanced_table.drop(columns="is_nat")
y = balanced_table["is_nat"].astype("int8")
groups = X.index.get_level_values(src_ip_field)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)
train_positions, test_positions = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_positions], X.iloc[test_positions]
y_train, y_test = y.iloc[train_positions], y.iloc[test_positions]

train_ips = set(X_train.index.get_level_values(src_ip_field))
test_ips = set(X_test.index.get_level_values(src_ip_field))
assert train_ips.isdisjoint(test_ips)

pd.DataFrame(
    {
        "train": y_train.value_counts().sort_index(),
        "test": y_test.value_counts().sort_index(),
    }
).rename(index={0: "non-NAT", 1: "NAT"})

## 6. Train and evaluate

The requested tree depth is fixed at 8. F1 and recall treat NAT as the positive class. In the confusion matrix, rows are true labels and columns are predicted labels.

In [ ]:
classifier.model.fit(X_train, y_train)
predictions = classifier.model.predict(X_test)

metrics = pd.Series(
    {
        "accuracy": accuracy_score(y_test, predictions),
        "f1": f1_score(y_test, predictions, pos_label=1),
        "recall": recall_score(y_test, predictions, pos_label=1),
    },
    name="score",
)
confusion = pd.DataFrame(
    confusion_matrix(y_test, predictions, labels=[0, 1]),
    index=["actual_non_nat", "actual_nat"],
    columns=["predicted_non_nat", "predicted_nat"],
)
display(metrics.to_frame(), confusion)

In [ ]:
pd.Series(
    classifier.model.feature_importances_,
    index=X.columns,
    name="importance",
).sort_values(ascending=False).to_frame()

## 7. Save and verify the runtime artifact

DAF expects a serialized `NATClassifier`, not a bare scikit-learn estimator. Saving the wrapper keeps the fitted tree, feature aggregation, positive label, and default threshold together. Only load trusted joblib files.

In [ ]:
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(classifier, model_path)

loaded_classifier = joblib.load(model_path)
if not isinstance(loaded_classifier, NATClassifier):
    raise TypeError("Saved artifact does not contain a NATClassifier")

example_window, example_ip = X_test.index[0]
example_flows = next(
    ip_dataset[example_ip]
    for window_start, ip_dataset in windowed_dataset
    if window_start == example_window and example_ip in ip_dataset
)
pd.testing.assert_series_equal(
    loaded_classifier.aggregate(example_flows),
    X_test.iloc[0],
    check_names=False,
)

print(f"Saved model: {model_path}")
print(f"Example decision and probability: {loaded_classifier.classify(example_flows)}")

Use the saved artifact in the DAF configuration:

```yaml
daf:
  windowing:
    enabled: true
    type: time
    size: 15min
    timestamp_field: TIME_FIRST
    consensus: majority

nat_detector:
  enabled: true
  path: auto
  model_path: ./detectors/nat_detector/final_models/decision_tree/nat_decision_tree_depth_8.joblib
  threshold: 0.5
  feature_mapping:
    SRC_IP: SRC_IP
    SRC_PORT: SRC_PORT
    DST_PORT: DST_PORT
    TCP_SYN_SIZE: TCP_SYN_SIZE
    TCP_WIN: TCP_WIN
    IP_TTL: IP_TTL
    BYTES: BYTES
    BYTES_REV: BYTES_REV
    PACKETS: PACKETS
    PACKETS_REV: PACKETS_REV
```

The mapping keys are the model's canonical raw fields and the values are input dataset columns. The runtime window size must remain 15 minutes to preserve the training boundary.